In [ ]:
import numpy as np
import pandas as pd

import kmeans_communes as km

OUT = km.OUT / "profil"
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200, "display.max_columns", 30)

GAMMES = [0, 100_000, 200_000, 350_000, np.inf]
GAMMES_LBL = ["< 100 k€", "100-200 k€", "200-350 k€", "> 350 k€"]

def niveau(valeur, ref, labels):
    """Place la valeur d'un cluster dans la distribution des communes (terciles)."""
    p = (ref < valeur).mean()
    return labels[0] if p < 1 / 3 else labels[1] if p < 2 / 3 else labels[2]

def sauver(nom, df, titre):
    print(f"\n=== {titre} ===")
    print(df.round(2).to_string())
    df.to_csv(OUT / f"{nom}.csv")

def main():
    c = pd.read_csv(km.OUT / "communes_clusters.csv", dtype={"code_insee": str, "code_dept": str})
    df = km.charger_mutations().merge(c[["code_insee", "cluster"]], on="code_insee", how="inner")

    # 1. Carte d'identité de chaque cluster + nom provisoire
    p = c.groupby("cluster").agg(
        n_communes=("code_insee", "size"), n_tx=("n_tx", "sum"),
        prix_m2=("prix_m2_med", "median"),
        prix_m2_q25=("prix_m2_med", lambda s: s.quantile(.25)),
        prix_m2_q75=("prix_m2_med", lambda s: s.quantile(.75)),
        ticket=("ticket_med", "median"), surface=("surf_med", "median"),
        tx_par_commune=("n_tx", "median"), dispersion=("dispersion", "median"))
    p["%_communes"] = 100 * p["n_communes"] / p["n_communes"].sum()
    p["%_transactions"] = 100 * p["n_tx"] / p["n_tx"].sum()
    p["nom_provisoire"] = [
        " · ".join([niveau(r.prix_m2, c["prix_m2_med"], ["prix bas", "prix moyen", "prix élevé"]),
                    niveau(r.tx_par_commune, c["n_tx"], ["peu liquide", "liquidité moyenne", "très liquide"]),
                    niveau(r.dispersion, c["dispersion"], ["prix homogènes", "dispersion moyenne", "prix très dispersés"])])
        for r in p.itertuples()]
    sauver("01_identite", p, "Identité des clusters (nom = terciles des communes, à reformuler)")

    # 2. Type de bien
    part = (pd.crosstab(df["cluster"], df["type_bien"], normalize="index") * 100)
    sauver("02_part_type_bien", part, "Part de chaque type de bien (% des transactions)")
    prix_type = df.pivot_table(index="cluster", columns="type_bien", values="prix_m2_fiable", aggfunc="median")
    sauver("03_prix_m2_par_type", prix_type, "Prix médian €/m² par type de bien")

    # 3. Gamme de prix
    df["gamme"] = pd.cut(df["valeur_fonciere"], GAMMES, labels=GAMMES_LBL)
    g = pd.crosstab(df["cluster"], df["gamme"], normalize="index") * 100
    sauver("04_gammes_de_prix", g, "Répartition par gamme de prix (% des transactions)")

    # 4. Localisation
    comp = pd.crosstab(c["region"], c["cluster"], normalize="columns") * 100
    sauver("05_composition_regionale", comp, "Composition de chaque cluster par région (% des communes du cluster)")
    prof_reg = pd.crosstab(c["region"], c["cluster"], normalize="index") * 100
    sauver("06_profil_par_region", prof_reg, "Profil de chaque région (% de ses communes dans chaque cluster)")

    # 5. Communes emblématiques (les plus grosses de chaque cluster)
    top = (c.sort_values("n_tx", ascending=False).groupby("cluster").head(8)
             .sort_values(["cluster", "n_tx"], ascending=[True, False])
             [["cluster", "commune", "code_dept", "region", "n_tx", "prix_m2_med", "ticket_med", "surf_med", "dispersion"]]
             .set_index("cluster"))
    sauver("07_communes_emblematiques", top, "Communes les plus actives de chaque cluster")

    # 6. Contrôle d'homogénéité : le cluster est-il compact ? (écart-type du prix/m² entre communes)
    homog = c.groupby("cluster")["prix_m2_med"].agg(["min", "median", "max", "std"])
    sauver("08_homogeneite_prix", homog, "Étendue du prix/m² médian par cluster (chevauchements = clusters mous)")

    print("\nFichiers écrits dans", OUT.resolve())

if __name__ == "__main__":
    main()